In [1]:
# ==============================================================================
# RESEARCH GRADE SCRIPT: HYBRID HIERARCHICAL MULTI-TASK NETWORK (HMTS-NET v2)
# BACKBONES: EfficientNet-B0 (Local) + ConvNeXt-Tiny (Global)
# FIX: Properly indexed Classifier layers to preserve Flatten() and LayerNorm()
# PLATFORM: Kaggle (with Auto-Resume Checkpointing)
# ==============================================================================

import os
import time
import json
import shutil
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

# Scikit-Learn for Evaluation Metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
class Config:
    BASE_DIR = "/kaggle/input/datasets/zahidhasantonmoy/new-dataset-fixing-sam-2-dataset-metadata/SAM2_GHAN_V30_Cleaned"
    TRAIN_DIR = f"{BASE_DIR}/dataset/train/images"
    VAL_DIR = f"{BASE_DIR}/dataset/val/images"
    TEST_DIR = f"{BASE_DIR}/dataset/test/images"
    META_CSV = f"{BASE_DIR}/metadata.csv"
    
    CKPT_DIR = "/kaggle/working/checkpoints"
    OUTPUT_DIR = "/kaggle/working/outputs"
    
    IMG_SIZE = 224
    BATCH_SIZE = 16  
    EPOCHS = 50
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 2
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    NUM_SPECIES = 204
    NUM_GENUS = 104
    NUM_VENOM = 5

os.makedirs(Config.CKPT_DIR, exist_ok=True)
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. DATASET PREPARATION & MAPPING
# ==========================================
df_meta = pd.read_csv(Config.META_CSV)
df_meta['class_name'] = df_meta['class_name'].str.lower().str.strip()

species_list = sorted(df_meta['class_name'].unique())
genus_list = sorted(df_meta['genus'].unique())
venom_list = sorted(df_meta['venom_category'].unique())

species2id = {name: idx for idx, name in enumerate(species_list)}
genus2id = {name: idx for idx, name in enumerate(genus_list)}
venom2id = {name: idx for idx, name in enumerate(venom_list)}

species_to_genus = dict(zip(df_meta['class_name'], df_meta['genus']))
species_to_venom = dict(zip(df_meta['class_name'], df_meta['venom_category']))

class HierarchicalSnakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths, self.species_labels, self.genus_labels, self.venom_labels = [], [], [], []
        
        valid_extensions = ('.jpg', '.jpeg', '.png')
        for species_folder in sorted(os.listdir(root_dir)):
            folder_path = os.path.join(root_dir, species_folder)
            if not os.path.isdir(folder_path): continue
                
            clean_species = species_folder.lower().strip()
            if clean_species not in species2id: continue
                
            sp_id = species2id[clean_species]
            ge_id = genus2id[species_to_genus[clean_species]]
            ve_id = venom2id[species_to_venom[clean_species]]
            
            for img_name in os.listdir(folder_path):
                if img_name.lower().endswith(valid_extensions):
                    self.image_paths.append(os.path.join(folder_path, img_name))
                    self.species_labels.append(sp_id)
                    self.genus_labels.append(ge_id)
                    self.venom_labels.append(ve_id)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform: image = self.transform(image)
        labels = {
            'species': torch.tensor(self.species_labels[idx], dtype=torch.long),
            'genus': torch.tensor(self.genus_labels[idx], dtype=torch.long),
            'venom': torch.tensor(self.venom_labels[idx], dtype=torch.long)
        }
        return image, labels

train_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(HierarchicalSnakeDataset(Config.TRAIN_DIR, train_transform), batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
val_loader = DataLoader(HierarchicalSnakeDataset(Config.VAL_DIR, val_test_transform), batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
test_loader = DataLoader(HierarchicalSnakeDataset(Config.TEST_DIR, val_test_transform), batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

# ==========================================
# 3. NOVEL ARCHITECTURE (Attentive Hybrid Fusion)
# ==========================================
class AttentiveFeatureFusion(nn.Module):
    def __init__(self, in_features, reduction=16):
        super(AttentiveFeatureFusion, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_features // reduction, in_features, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.attention(x)

class HybridHMTSNet(nn.Module):
    def __init__(self):
        super(HybridHMTSNet, self).__init__()
        
        # Branch 1: EfficientNet-B0
        self.effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        # Only replace the final Linear layer to preserve Dropout and internal flattening
        eff_features = self.effnet.classifier[1].in_features
        self.effnet.classifier[1] = nn.Identity()
        
        # Branch 2: ConvNeXt-Tiny
        self.convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        # Only replace the final Linear layer to preserve LayerNorm2d and Flatten
        conv_features = self.convnext.classifier[2].in_features
        self.convnext.classifier[2] = nn.Identity()
        
        # Fusion
        self.total_features = eff_features + conv_features
        self.fusion_attention = AttentiveFeatureFusion(self.total_features)
        self.dropout = nn.Dropout(p=0.4)
        
        # Multi-Task Heads
        self.species_head = nn.Linear(self.total_features, Config.NUM_SPECIES)
        self.genus_head = nn.Linear(self.total_features, Config.NUM_GENUS)
        self.venom_head = nn.Linear(self.total_features, Config.NUM_VENOM)

    def forward(self, x):
        f_eff = self.effnet(x)
        f_conv = self.convnext(x)
        
        # Ensure outputs are explicitly flattened just in case
        f_eff = torch.flatten(f_eff, 1)
        f_conv = torch.flatten(f_conv, 1)
        
        fused = torch.cat((f_eff, f_conv), dim=1)
        
        fused = self.fusion_attention(fused)
        fused = self.dropout(fused)
        
        out_sp = self.species_head(fused)
        out_ge = self.genus_head(fused)
        out_ve = self.venom_head(fused)
        return out_sp, out_ge, out_ve

model = HybridHMTSNet().to(Config.DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = torch.cuda.amp.GradScaler() 

# ==========================================
# 4. TRAINING UTILS
# ==========================================
def calculate_accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
        res.append(correct_k.mul_(100.0 / batch_size).item())
    return res

def save_checkpoint(state, is_best, filename="checkpoint.pth"):
    ckpt_path = os.path.join(Config.CKPT_DIR, filename)
    torch.save(state, ckpt_path)
    if is_best: shutil.copyfile(ckpt_path, os.path.join(Config.CKPT_DIR, "model_best.pth"))

def load_checkpoint():
    ckpt_path = os.path.join(Config.CKPT_DIR, "checkpoint.pth")
    if os.path.isfile(ckpt_path):
        print(f"[*] Resuming from checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path)
        model.load_state_dict(ckpt['state_dict'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        return ckpt['epoch'], ckpt['best_val_acc'], ckpt['history']
    return 0, 0.0, {'train_loss':[], 'val_loss':[], 'val_acc1':[], 'val_acc5':[]}

# ==========================================
# 5. MAIN TRAINING LOOP
# ==========================================
start_epoch, best_val_acc, history = load_checkpoint()

print(f"========== STARTING SOTA HYBRID TRAINING ON {Config.DEVICE} ==========")
for epoch in range(start_epoch, Config.EPOCHS):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [TRAIN]")
    for images, labels in pbar:
        images = images.to(Config.DEVICE)
        lbl_sp = labels['species'].to(Config.DEVICE)
        lbl_ge = labels['genus'].to(Config.DEVICE)
        lbl_ve = labels['venom'].to(Config.DEVICE)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            out_sp, out_ge, out_ve = model(images)
            loss_sp = criterion(out_sp, lbl_sp)
            loss_ge = criterion(out_ge, lbl_ge)
            loss_ve = criterion(out_ve, lbl_ve)
            loss = loss_sp + (0.3 * loss_ge) + (0.3 * loss_ve) 
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        pbar.set_postfix({'Loss': f"{loss.item():.4f}", 'LR': f"{scheduler.get_last_lr()[0]:.6f}"})
        
    train_loss = running_loss / len(train_loader)
    scheduler.step()
    
    # Validation Loop
    model.eval()
    val_loss, top1_acc, top5_acc = 0.0, 0.0, 0.0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [VAL]"):
            images = images.to(Config.DEVICE)
            lbl_sp = labels['species'].to(Config.DEVICE)
            
            with torch.cuda.amp.autocast():
                out_sp, _, _ = model(images)
                loss_sp = criterion(out_sp, lbl_sp)
            
            val_loss += loss_sp.item()
            acc1, acc5 = calculate_accuracy(out_sp, lbl_sp, topk=(1, 5))
            top1_acc += acc1
            top5_acc += acc5

    val_loss /= len(val_loader)
    top1_acc /= len(val_loader)
    top5_acc /= len(val_loader)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc1'].append(top1_acc)
    history['val_acc5'].append(top5_acc)
    
    print(f"\n[Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Top-1: {top1_acc:.2f}% | Val Top-5: {top5_acc:.2f}%\n")
    
    is_best = top1_acc > best_val_acc
    best_val_acc = max(top1_acc, best_val_acc)
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_val_acc': best_val_acc,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'history': history
    }, is_best)

# ==========================================
# 6. EVALUATION ON TEST SET (FULL METRICS)
# ==========================================
print("\n========== FINAL EVALUATION (TEST FOLDER) ==========")
model.load_state_dict(torch.load(os.path.join(Config.CKPT_DIR, "model_best.pth"))['state_dict'])
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(Config.DEVICE)
        lbl_sp = labels['species'].cpu().numpy()
        
        with torch.cuda.amp.autocast():
            out_sp, _, _ = model(images)
            
        probs = torch.softmax(out_sp, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        
        all_labels.extend(lbl_sp)
        all_preds.extend(preds)
        all_probs.extend(probs)

# 1. Classification Report
report = classification_report(all_labels, all_preds, target_names=species_list, zero_division=0)
with open(os.path.join(Config.OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

# 2. Confusion Matrix
plt.figure(figsize=(40, 40))
sns.heatmap(confusion_matrix(all_labels, all_preds), cmap='Blues', xticklabels=False, yticklabels=False)
plt.title('Species Confusion Matrix (204 Classes)')
plt.savefig(os.path.join(Config.OUTPUT_DIR, "confusion_matrix.png"), dpi=300, bbox_inches='tight')
plt.close()

# 3. Learning Curves
epochs_range = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss')
plt.plot(epochs_range, history['val_loss'], label='Val Loss')
plt.legend(); plt.title('Loss Curve')
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['val_acc1'], label='Top-1 Acc')
plt.plot(epochs_range, history['val_acc5'], label='Top-5 Acc')
plt.legend(); plt.title('Accuracy Curve')
plt.savefig(os.path.join(Config.OUTPUT_DIR, "training_curves.png"), dpi=300)
plt.close()

# 4. ROC-AUC Macro Average
try:
    bin_labels = label_binarize(all_labels, classes=range(Config.NUM_SPECIES))
    macro_roc_auc = roc_auc_score(bin_labels, all_probs, average="macro", multi_class="ovr")
    print(f"\n✅ Final Test Macro ROC-AUC Score: {macro_roc_auc:.4f}")
    with open(os.path.join(Config.OUTPUT_DIR, "roc_auc_score.txt"), "w") as f:
        f.write(f"ROC-AUC Macro: {macro_roc_auc:.4f}")
except Exception as e:
    print("ROC calculation error:", e)

print(f"\n🚀 SOTA TRAINING COMPLETE! Results saved to {Config.OUTPUT_DIR}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 130MB/s] 


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 179MB/s]


========== STARTING SOTA HYBRID TRAINING ON cuda ==========


...


[Epoch 49] Train Loss: 1.2785 | Val Loss: 2.4369 | Val Top-1: 65.89% | Val Top-5: 83.45%



Epoch 50/50 [VAL]: 100%|██████████| 273/273 [00:21<00:00, 12.96it/s]



[Epoch 50] Train Loss: 1.2762 | Val Loss: 2.4521 | Val Top-1: 65.16% | Val Top-5: 83.29%


========== FINAL EVALUATION (TEST FOLDER) ==========


Testing: 100%|██████████| 273/273 [00:34<00:00,  7.93it/s]



✅ Final Test Macro ROC-AUC Score: 0.9824

🚀 SOTA TRAINING COMPLETE! Results saved to /kaggle/working/outputs
